# Parameter estimation — PEMFC model calibration with cross-validation

This notebook demonstrates `marapendi`'s parameter-estimation workflow using
`SteadyStatePolarizationCurveCalibration`, the main class for fitting model
parameters to polarisation-curve and HFR measurements.

The workflow has three stages:

1. **Global sensitivity analysis** (`compute_global_sensitivity`) — Sobol-based
   screening ranks all 38 free parameters by their influence on the model output.
   Results are cached to a `.npz` file and reloaded for inspection.
2. **k-fold cross-validation vs. complexity**
   (`run_k_fold_cross_validation_vs_complexity`) — sweeps the number of estimated
   parameters (1 to 38), refitting the model at each complexity level in a
   leave-one-condition-out scheme. The 1-SE rule selects the simplest model within
   one standard error of the minimum RMSE.
3. **Diagnostics** — RMSE vs. complexity curves, parameter evolution plots,
   cross-validation polarisation curves, and simulated internal variables for the
   optimal n = 18 model.

The model here is `ExplicitSteadyStateModel` with
`MembraneWaterBalanceModelPiecewise` (piecewise-linear RH(λ) approximation,
characterised in `07_picewise_linear_approximation_water_conent.ipynb`).
Experimental data (9 conditions, MEA62) are embedded inline as a CSV string.

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import marapendi as mrpd
from marapendi.estimation.parameters import Parameter, UnknownParameter

plt.rcParams.update({"text.usetex": False})

In [ ]:
filename = 'test'

In [ ]:
_exp_data_csv = """case,current-density,voltage,hfr
1,0,0.911451,
1,0.08,0.819964,0.0413076
1,0.2,0.783851,
1,0.4,0.746728,0.0397381
1,0.6,0.7192,
1,0.8,0.691927,
1,1,0.666024,0.0369162
1,1.2,0.637758,
1,1.4,0.609555,
1,1.56,0.587704,
2,0,0.946503,
2,0.08,0.861721,0.0386086
2,0.2,0.823163,
2,0.4,0.782723,0.0377255
2,0.6,0.750069,
2,0.8,0.719233,
2,1,0.687873,0.0364815
2,1.2,0.656312,
2,1.4,0.626726,
2,1.56,0.605712,
2,1.59,0.599881,
3,0,0.919656,
3,0.08,0.814908,0.04987
3,0.2,0.772305,
3,0.4,0.730443,0.0554072
3,0.6,0.69555,
3,0.8,0.664588,
3,1,0.636433,0.0528002
3,1.2,0.610458,
3,1.4,0.583257,
3,1.56,0.549974,
3,1.8,0.494066,
3,1.96,0.436816,
4,0,0.929941,
4,0.08,0.851711,0.0368974
4,0.2,0.816085,
4,0.4,0.785446,0.0374853
4,0.6,0.760909,
4,0.8,0.740034,
4,1,0.720167,0.0365731
4,1.2,0.701043,
4,1.4,0.679064,
4,1.56,0.657917,
4,1.8,0.619024,
4,2,0.578545,
5,0,0.927345,
5,0.08,0.843686,0.0363732
5,0.2,0.807182,
5,0.4,0.774042,0.037437
5,0.6,0.75205,
5,0.8,0.729708,
5,1,0.707576,0.0338028
5,1.2,0.684814,
5,1.4,0.661233,
5,1.56,0.642694,
5,1.79,0.599711,
6,0,0.922887,
6,0.08,0.810585,0.0478001
6,0.2,0.774263,
6,0.4,0.737869,0.0453726
6,0.6,0.70696,
6,0.8,0.678401,
6,1,0.651822,0.0446186
6,1.2,0.626913,
6,1.4,0.596979,
6,1.56,0.566364,
7,0,0.929048,
7,0.08,0.829254,0.0376741
7,0.2,0.790205,
7,0.4,0.752194,0.0370008
7,0.6,0.722174,
7,0.8,0.693328,
7,1,0.664069,0.0362274
7,1.2,0.635142,
7,1.4,0.606177,
7,1.56,0.583332,
8,0,0.919067,
8,0.08,0.826517,0.0333321
8,0.2,0.791053,
8,0.4,0.757887,0.0318667
8,0.6,0.733209,
8,0.8,0.711132,
8,1,0.689397,0.0322763
8,1.16,0.672052,
8,1.4,0.640206,
8,1.56,0.611004,
8,1.6,0.600006,
9,0,0.91687,
9,0.08,0.823573,0.0330154
9,0.2,0.782671,
9,0.4,0.748927,0.032455
9,0.6,0.722018,
9,0.8,0.695022,
9,1,0.66496,0.0322796
9,1.2,0.631988,
9,1.4,0.596083,
9,1.56,0.565461,
"""

_exp_data_df = pd.read_csv(pd.io.common.StringIO(_exp_data_csv))
_exp_data_df['current-density'] *= 1e4
_exp_data_df['hfr'] *= 1e-4

In [ ]:
_exp_data_df['case']

## Experimental data

Nine polarisation-curve and HFR measurement sets for MEA62, spanning temperatures
from 50 to 90 °C, pressures from 1.5 to 2.5 bar, and relative humidities from 30
to 80 %. The data are embedded as a CSV string — no external file is required.
`SteadyStatePolarizationCurveCalibration` accepts them as a `pd.DataFrame` with
columns `case`, `current-density`, `voltage`, and `hfr`.

## Fuel-cell model with piecewise-linear membrane water balance

`create_cell(params)` builds a `FuelCell` from a parameter dict — the standard
pattern for parameter estimation, where `SteadyStatePolarizationCurveCalibration`
calls this function at each evaluation with updated parameter values.

The calibration object wraps the model, data, and parameter definitions, and
exposes the full estimation API:
- `compute_global_sensitivity(m=12, ...)` — Saltelli sampling, Sobol indices
- `run_k_fold_cross_validation_vs_complexity(n_params_list, ...)` — complexity sweep
- `load_cross_validation_results(filename)` — reload cached results for plotting

In [ ]:
def create_cell(params):
    membrane = mrpd.PFSA(
        ionomer=mrpd.PFSAIonomer(
            equivalent_weight=params['memb-equiv-weight'],
            dry_density=2000.,
            conductivity_exp=params['memb-cond-exp'],
            conductivity_activation_energy=params['memb-E-act-cond'],
            conductivity_correction=params['memb-cond-correction'],
            reference_water_diffusivity=params['memb-water-diff'],
            reference_water_absorption_coefficient=params['memb-abs-constant'],
            water_diffusivity_activation_energy=params['E-act-memb-diff'],
            water_absorption_activation_energy=params['E-act-memb-abs'],
        ),
        dry_thickness=params['memb-thickness'],
    )

    orr_kinetics = mrpd.ElectrochemicalReaction(
        reference_exchange_current_density=params['i0-c'],
        reaction_order=params['gamma-c'],
        activation_energy=params['E-act-ca'],
        reference_activity=1.01325e5,
        reference_temperature=353.15,
        number_of_electrons=1,
        charge_transfer_coeff=params['alpha-c'],
    )

    liq_model = mrpd.DarcyTransportModel(J_function_exponent=params['wet-transition'])

    gdl = {
        side: mrpd.GasDiffusionLayer(
            thickness=params['gdl-thickness'],
            contact_angle=params['gdl-theta'],
            effective_gas_diffusion_ratio=params['gdl-eff-diff-ratio'],
            absolute_permeability=params['gdl-abs-perm'],
            porosity=params['gdl-porosity'],
            thermal_conductivity=params['gdl-thermal-cond'],
            two_phase_transport_model=liq_model,
            transport_resistance_model=mrpd.PorousGasDiffusionModel(
                water_saturation_exponent=params['n_s']),
        ) for side in ['ca', 'an']
    }

    ch = {
        side: mrpd.FlowChannel(
            height=params['ch-height'], width=1e-3, n_parallel=1, length=21 * 50e-3,
            reactant='o2' if side == 'ca' else 'h2',
            transport_resistance_model=mrpd.ChannelGasResistanceModel(
                sherwood=params['Sh'], B_ch=params['B_ch']),
        ) for side in ['an', 'ca']
    }

    ionomer = mrpd.PFSAIonomer(
        conductivity_correction=params['ionomer-cond-corr'],
        conductivity_exp=params['ionomer-cond-exp'],
        conductivity_activation_energy=params['ionomer-E-act-cond'],
    )

    ca_cl = mrpd.PtCCatalystLayer(
        ecsa=params['ecsa'], platinum_loading=params['pt-loading'], ionomer=ionomer,
        catalyst_platinum_weight_percent=params['pt-wt-percent'],
        ionomer_to_carbon_ratio=params['ic-ratio'],
        ionomer_k1=params['ionomer-k1'], ionomer_k2=params['ionomer-k2'],
        ionomer_k3=params['ionomer-k3'],
        pore_diameter=params['cl-pore-diameter'], omega_PtO=0,
        carbon_agglomerate_radius=params['radius-carbon'],
        thickness=params['pt-loading'] * 2.8e-6 / 0.1e-2,
        absolute_permeability=params['cl-abs-perm'], contact_angle=params['cl-theta'],
        thermal_conductivity=params['cl-thermal-cond'], reaction=orr_kinetics,
        two_phase_transport_model=liq_model,
        transport_resistance_model=mrpd.PorousGasDiffusionModel(water_saturation_exponent=1.5),
    )

    an_cl = mrpd.PtCCatalystLayer(
        ecsa=params['ecsa'], platinum_loading=1e-3,
        catalyst_platinum_weight_percent=params['pt-wt-percent'],
        ionomer_to_carbon_ratio=params['ic-ratio'], ionomer=ionomer,
        pore_diameter=params['cl-pore-diameter'],
        carbon_agglomerate_radius=params['radius-carbon'],
        thickness=2.8e-6, absolute_permeability=params['cl-abs-perm'],
        contact_angle=params['cl-theta'],
        thermal_conductivity=params['cl-thermal-cond'],
        two_phase_transport_model=liq_model,
        transport_resistance_model=mrpd.PorousGasDiffusionModel(water_saturation_exponent=1.5),
    )

    return mrpd.FuelCell(
        electrical_resistance=params['elec-resistance'],
        area=25e-4,
        ca=mrpd.FuelCellSide(
            cl=ca_cl, gdl=gdl['ca'], has_gdl=True, has_mpl=False, ch=ch['ca'],
            thermal_contact_resistance=params['tcr'],
        ),
        an=mrpd.FuelCellSide(
            cl=an_cl, gdl=gdl['an'], has_gdl=True, has_mpl=False, ch=ch['an'],
            thermal_contact_resistance=params['tcr'],
        ),
        membrane=membrane,
    )

In [ ]:
conditions_df = pd.DataFrame({
    'case':        [1,2,3,4,5,6,7,8,9],
    'cell-temperature': np.array([80, 50, 80, 80, 80, 90, 50, 80, 80]) + 273.15,
    'pressure-an': np.array([1.5, 2.5, 1.5, 2.5, 2.5, 1.5, 1.5, 1.5, 1.5]) * 1e5,
    'pressure-ca': np.array([1.5, 2.5, 1.5, 2.5, 2.3, 1.5, 1.5, 1.5, 1.5]) * 1e5,
    'rh-an':       np.array([50, 50, 30, 30, 50, 50, 80, 80, 80]) / 100,
    'rh-ca':       np.array([50, 50, 30, 30, 30, 50, 80, 80, 80]) / 100,
    'st-an':       [1.5, 1.5, 1.2, 2.0, 1.5, 1.5, 1.5, 2.0, 1.5],
    'st-ca':       [2.0, 2.0, 2.5, 2.5, 2.0, 2.5, 2.0, 2.5, 1.5],
})

In [ ]:
unknown_parameters = [
    # ORR kinetics - 4 params
    UnknownParameter(value=2.54e-4, key='i0-c',     symbol=r'$i_{0,ca}$',                          units='A/cm² Pt',    factor=1,    initial_guess=2.54e-4, lower_bound=1e-5,   upper_bound=1e-2,   is_linear=False),
    UnknownParameter(value=0.54,    key='gamma-c',  symbol=r'$\gamma_{ca}$',                       units='n.d.',        factor=1,    initial_guess=0.54,    lower_bound=0.5,    upper_bound=1.),
    UnknownParameter(value=1.,      key='alpha-c',  symbol=r'$\alpha_{ca}$',                       units='n.d.',        factor=1,    initial_guess=1.,      lower_bound=0.5,    upper_bound=1.),
    UnknownParameter(value=67e6,    key='E-act-ca', symbol='$E_{act,ca}$',                         units='kJ/mol',      factor=1e6,  initial_guess=67e6,    lower_bound=30e6,   upper_bound=80e6),

    # CL properties - 7 params
    UnknownParameter(value=.3e-2,   key='pt-loading',       symbol=r'$L_{\text{Pt},ca}$',          units='mg/cm²',      factor=1e-2, initial_guess=.3e-2,   lower_bound=0.2e-2, upper_bound=0.5e-2),
    UnknownParameter(value=60e3,    key='ecsa',             symbol='ECSA',                         units='m²/g',        factor=1e3,  initial_guess=60e3,    lower_bound=40e3,   upper_bound=90e3),
    UnknownParameter(value=1.4,     key='ic-ratio',         symbol='IC$_{ca}$',                    units='n.d.',        factor=1,    initial_guess=1.4,     lower_bound=0.5,    upper_bound=1.4),
    UnknownParameter(value=25e-9,   key='radius-carbon',    symbol='$r_C$',                        units='nm',          factor=1e-9, initial_guess=25e-9,   lower_bound=10e-9,  upper_bound=40e-9),
    UnknownParameter(value=1e-13,   key='cl-abs-perm',      symbol=r'$K^{abs}_{\text{CL}}$',       units='e-12 m²',     factor=1e-12,initial_guess=1e-13,   lower_bound=1e-14,  upper_bound=1e-12,  is_linear=False),
    UnknownParameter(value=40e-9,   key='cl-pore-diameter', symbol=r'$d_{p,\text{CL}}$',           units='nm',          factor=1e-9, initial_guess=40e-9,   lower_bound=10e-9,  upper_bound=70e-9),
    UnknownParameter(value=97.,     key='cl-theta',         symbol=r'$\theta_{\text{CL}}$',        units='deg',         factor=1,    initial_guess=97.,     lower_bound=92,     upper_bound=110),

    # Ionomer properties
    UnknownParameter(value=8.5,     key='ionomer-k1',        symbol='$k_1$',                       units='n.d.',        factor=1,    initial_guess=8.5,     lower_bound=2,      upper_bound=10),
    UnknownParameter(value=5.4,     key='ionomer-k2',        symbol='$k_2$',                       units='n.d.',        factor=1,    initial_guess=5.4,     lower_bound=2,      upper_bound=10),
    UnknownParameter(value=5.4,     key='ionomer-k3',        symbol='$k_3$',                       units='n.d.',        factor=1,    initial_guess=5.4,     lower_bound=2,      upper_bound=10),
    UnknownParameter(value=1,       key='ionomer-cond-corr', symbol=r'$\xi_{\sigma}^{ion}$',       units='n.d.',        factor=1,    initial_guess=1,       lower_bound=0.1,    upper_bound=50,     is_linear=False),
    UnknownParameter(value=1.5,     key='ionomer-cond-exp',  symbol=r'$n_{\sigma}^{ion}$',         units='n.d.',        factor=1,    initial_guess=1.5,     lower_bound=1,      upper_bound=2),
    UnknownParameter(value=15e6,    key='ionomer-E-act-cond',symbol=r'$E^{ion}_{act,\sigma}$',     units='kJ/mol',      factor=1e6,  initial_guess=15e6,    lower_bound=5e6,    upper_bound=30e6),

    # Membrane properties
    UnknownParameter(value=1.0,     key='memb-cond-correction',symbol=r'$\xi_{\sigma}^{mb}$',      units='n.d.',        factor=1,    initial_guess=1.0,     lower_bound=0.1,    upper_bound=50,     is_linear=False),
    UnknownParameter(value=1.5,     key='memb-cond-exp',     symbol=r'$n_{\sigma}^{mb}$',          units='n.d.',        factor=1,    initial_guess=1.5,     lower_bound=1,      upper_bound=2),
    UnknownParameter(value=15e6,    key='memb-E-act-cond',   symbol=r'$E^{mb}_{act,\sigma}$',      units='kJ/mol',      factor=1e6,  initial_guess=15e6,    lower_bound=5e6,    upper_bound=30e6),
    UnknownParameter(value=12e-6,   key='memb-thickness',    symbol=r'$\delta_{mb}$',               units='μm',          factor=1e-6, initial_guess=12e-6,   lower_bound=8e-6,   upper_bound=15e-6),
    UnknownParameter(value=1100.,   key='memb-equiv-weight', symbol='EW',                           units='g/mol',       factor=1,    initial_guess=1100.,   lower_bound=700.,   upper_bound=1100.),
    UnknownParameter(value=2e-10,   key='memb-water-diff',   symbol=r'$D_{\lambda}$',               units='1e-10 m²/s',  factor=1e-10,initial_guess=2e-10,   lower_bound=1e-10,  upper_bound=10e-10),
    UnknownParameter(value=1e-5,    key='memb-abs-constant', symbol='$k_{abs}$',                    units='m/s',         factor=1,    initial_guess=1e-5,    lower_bound=1e-6,   upper_bound=1e-4,   is_linear=False),
    UnknownParameter(value=20e6,    key='E-act-memb-diff',   symbol=r'$E^{mb}_{act,D_\lambda}$',   units='kJ/mol',      factor=1e6,  initial_guess=20e6,    lower_bound=15e6,   upper_bound=30e6),
    UnknownParameter(value=20e6,    key='E-act-memb-abs',    symbol=r'$E^{mb}_{act,abs}$',          units='kJ/mol',      factor=1e6,  initial_guess=20e6,    lower_bound=15e6,   upper_bound=30e6),

    # GDL properties - 5 params
    UnknownParameter(value=150e-6,  key='gdl-thickness',     symbol=r'$\delta_{\text{GDL}}$',       units='μm',          factor=1e-6, initial_guess=150e-6,  lower_bound=100e-6, upper_bound=200e-6),
    UnknownParameter(value=0.3,     key='gdl-eff-diff-ratio',symbol=r'$D_{\text{GDL}}^{\mathit{eff}}/D$', units='n.d.', factor=1,    initial_guess=0.3,     lower_bound=0.05,   upper_bound=0.5),
    UnknownParameter(value=1e-12,   key='gdl-abs-perm',      symbol=r'$K^{abs}_{\text{GDL}}$',      units='e-12 m²',     factor=1e-12,initial_guess=1e-12,   lower_bound=1e-13,  upper_bound=1e-11,  is_linear=False),
    UnknownParameter(value=0.5,     key='gdl-thermal-cond',  symbol=r'$k_{\text{GDL}}$',            units='W/mK',        factor=1,    initial_guess=0.5,     lower_bound=0.05,   upper_bound=2,      is_linear=False),
    UnknownParameter(value=120.,    key='gdl-theta',         symbol=r'$\theta_{\text{GDL}}$',        units='deg',         factor=1,    initial_guess=120.,    lower_bound=100,    upper_bound=150),

    # Other - 6 params
    UnknownParameter(value=4.4e-4,  key='tcr',               symbol='TCR',                          units='1e-4 Kcm²/W', factor=1e-4, initial_guess=4.4e-4,  lower_bound=0,      upper_bound=10e-4),
    UnknownParameter(value=33e-7,   key='elec-resistance',   symbol='$r_{elec}$',                   units='mΩ·cm',       factor=1e-7, initial_guess=33e-7,   lower_bound=10e-7,  upper_bound=50e-7),
    UnknownParameter(value=3.6,     key='Sh',                symbol='$Sh$',                         units='n.d.',        factor=1,    initial_guess=3.6,     lower_bound=0.01,   upper_bound=10,     is_linear=False),
    UnknownParameter(value=1.,      key='B_ch',              symbol=r'$\xi_{\text{CH}}$',           units='n.d.',        factor=1,    initial_guess=1.,      lower_bound=0,      upper_bound=5),
    UnknownParameter(value=0.4,     key='wet-transition',    symbol='$n_{J}$',                      units='n.d.',        factor=1,    initial_guess=0.4,     lower_bound=0.1,    upper_bound=3),
    UnknownParameter(value=2,       key='n_s',               symbol='$n_s$',                        units='n.d.',        factor=1,    initial_guess=2,       lower_bound=1.5,    upper_bound=3),
]

known_parameters = [
    Parameter(value=0.6,   key='gdl-porosity',         symbol=r'$\varepsilon_{\text{GDL}}$', units='n.d.',      factor=1),
    Parameter(value=0.22,  key='cl-thermal-cond',      symbol=r'$k_{\text{CL}}$',            units='W/mK',      factor=1),
    Parameter(value=1,     key='alpha-w',              symbol=r'$\alpha_w$',                 units='n.d.',      factor=1),
    Parameter(value=0.4,   key='pt-wt-percent',        symbol=r'$w_{Pt}$',                   units='n.d.',      factor=1),
    Parameter(value=1e-3,  key='ch-height',            symbol=r'$h_{ch}$',                   units='mm',        factor=1e-3),
]

In [ ]:
pe = mrpd.SteadyStatePolarizationCurveCalibration(
    conditions_dataset=conditions_df, 
    experimental_dataset = _exp_data_df,
    cell_creator=create_cell,
    unknown_parameters = unknown_parameters, 
    known_parameters = known_parameters,
    cell_model=mrpd.ExplicitSteadyStateModel(
        water_balance_model=mrpd.WaterBalanceModel(
            membrane_water_balance_model=mrpd.MembraneWaterBalanceModelPiecewise()
        )
    )
)

In [ ]:
recompute_sensitivity = True
if recompute_sensitivity: 
    pe.compute_global_sensitivity(
        m=12,
        check_samples=True,
        rmse_limit=0.050,
        filename_to_save=f'sensitivity_results_{filename}',
    )

## Global sensitivity analysis

`plot_parameter_ranking` sorts parameters by their Sobol total-effect index —
high-ranked parameters dominate model output variance and are estimated first.
`plot_global_sensitivity` shows the raw index values on a log scale;
`plot_colinearity_map` reveals pairs of parameters that cannot be identified
independently from this dataset (high colinearity → do not estimate both).

In [ ]:
pe.load_global_sensitivity_results(f'sensitivity_results_{filename}.npz')
fig, ax, ax2 = mrpd.plot_parameter_ranking(pe)


In [ ]:
fig1, ax1 = mrpd.plot_global_sensitivity(pe, figsize=(12, 3))
ax1.set_ylabel('Normalized sensitivity')
ax1.grid()
ax1.set_yticks([1e-6, 1e-3, 0.1, 1e1])

In [ ]:
fig2, ax2 = mrpd.plot_colinearity_map(pe, xlabel_angle=70, cmap='Blues',figsize=(13,10))
for text in ax2.texts: 
    text.set_fontsize(6)

In [ ]:
pe.set_k_folds(9)

In [ ]:
pe.run_k_fold_cross_validation_vs_complexity(n_params_list=[i for i in range(1,20)], 
                               force_restart=True,


                               filename='test')
pe.reset_unknown_parameters()


In [ ]:
pe.reset_unknown_parameters()

## Estimation results — RMSE vs. complexity and cross-validation curves

`plot_rmse_vs_complexity` shows the leave-one-out test RMSE (median ± IQR across
folds) as a function of the number of estimated parameters for both cell voltage
and HFR. The 1-SE rule selects n = 18 as the preferred complexity.

`plot_cross_validation_curves` overlays simulated and measured polarisation curves
for each fold at the chosen complexity, giving a visual check of generalisation
quality. `plot_parameter_vs_complexity` tracks how each estimated parameter evolves
as complexity grows.

In [ ]:
cv_results = pe.load_cross_validation_results(filename='test')
cv_results


In [ ]:
len(pe.unknown_p_list)

In [ ]:
fig, ax, voltage_rmse_df, _ = mrpd.plot_rmse_vs_complexity(
    pe,
    cv_results,
    ylabel='Cell voltage\nRMSE (mV)',
    quantity_multiplier=1000,
    use_median=True,
    figsize=(9.5, 2.7),
    xrotation=70,
    dpi=300
)
ax.semilogy()
ax.set_ylim([5,800])

In [ ]:
fig, ax, hfr_rmse_df, _ = mrpd.plot_rmse_vs_complexity(
    pe, 
    cv_results,
    ylabel='HFR RMSE\n(mΩ·cm²)',
    variable='hfr',
    quantity_multiplier=1e7,
    use_median=True,
    figsize=(9.5, 2.7),
    xrotation=70,
    dpi=300
)
ax.semilogy()
ax.set_ylim([0.5,1200])

In [ ]:
voltage_rmse_df

In [ ]:
voltage_rmse_stats_df, voltage_rmse_latex = mrpd.rmse_complexity_table(voltage_rmse_df)
print(voltage_rmse_latex)

hfr_rmse_stats_df, hfr_rmse_latex = mrpd.rmse_complexity_table(hfr_rmse_df)
print(hfr_rmse_latex)

In [ ]:
fig, ax = mrpd.plot_parameter_vs_complexity(
    pe, 
    cv_results,
    n_cols=4,
    figsize=(11, 12),
    save_path=None,
    dpi=300,
)

In [ ]:
pe.build_cases_conditions(current_density=np.linspace(1, 2.2e4, 100))
fig, ax = mrpd.plot_cross_validation_curves(
    pe,
    cv_results,
    18, 
    variable='voltage',
    quantity_symbol=r"$V_{cell}$",
    quantity_unit="V",
    x_label=r"$i$ (A/cm$^2$)",
    save_path=None,
    dpi=300,
    uncertainty=0.05
)
ax[0,0].set_ylim(0.35,1)

## Internal variables at n = 18

Simulates the full `CellState` for all 9 conditions at the n = 18 estimated
parameter set and plots five rows of diagnostics: cell voltage, HFR, cathode
CL liquid saturation, cathode water fluxes (total, liquid, membrane), and ionomer
water content across the MEA (cathode CL → membrane → anode CL). All quantities
are accessible as fields of the returned `CellState` object.

In [ ]:
n_params = 18
fold_id = 2

fig, ax = plt.subplots(figsize=(12, 8), nrows=5, ncols=len(pe.full_case_list),
                       sharex=True, sharey='row')

fold_results = cv_results[(cv_results.n_params == n_params) & (cv_results.fold_id == fold_id)]
voltage_cases, hfr_cases, state_cases = pe.simulate_for_fold_results(fold_results)

for k, case in enumerate(pe.full_case_list):
    state = state_cases[case]
    i_sim = state.current_density * 1e-4
    U_sim = state.cell_voltage
    case_dataset = pe.get_case_dataset(case)

    ax[0, k].plot(i_sim, U_sim, f'C{k}', label='Sim.')
    ax[0, k].fill_between(i_sim, U_sim-.02, U_sim+.02, color=f'C{k}', alpha=0.3, label='± 20 mV')
    ax[0, k].plot(case_dataset['current-density'] * 1e-4, case_dataset['voltage'],
                  's' + f'C{k}', markersize=2.5, alpha=0.5, label='Exp.')
    ax[0, k].set_ylim([0.4, 0.95])

    ax[1, k].plot(i_sim, hfr_cases[case] * 1e7, f'C{k}', label='Sim.')
    ax[1, k].plot(case_dataset['current-density'] * 1e-4, case_dataset['hfr'] * 1e7,
                  's' + f'C{k}', markersize=2.5, alpha=0.5, label='Exp.')
    ax[1, k].set_ylim([30, 60])
    ax[2, k].plot(i_sim, 100 * state.ca.cl.liquid_saturation, '--' + f'C{k}', label=r'$s_l^{ca}$')
    # ax[2, k].plot(i_sim, state.ca.membrane, '.' + f'C{k}', label=r'$s_l^{ca}$')
    # ax[2, k].set_ylim([0, 115])

    ax[3, k].plot(i_sim, 1e6 * state.ca.membrane_water_flux,  '-'  + f'C{k}', label='$J_{mb}^{ca}$')
    ax[3, k].plot(i_sim, 1e6 * state.ca.liquid_flux,          '--' + f'C{k}', label='$J_l^{ca}$')
    ax[3, k].plot(i_sim, 1e6 * state.ca.water_flux,           '-.' + f'C{k}', label='$J_w^{ca}$')

    ax[4, k].plot(i_sim, state.ca.cl.ionomer_water_content, '-.' + f'C{k}', label=r'$\lambda^{ca}_{ion,\text{CL}}$')
    ax[4, k].plot(i_sim, state.ca.membrane_interface_water_content,      '--'  + f'C{k}', label=r'$\lambda^{ca}$')
    ax[4, k].plot(i_sim, state.membrane.water_content,      '-'  + f'C{k}', label=r'$\lambda^{avg}_{mb}$')
    ax[4, k].plot(i_sim, state.an.membrane_interface_water_content,      '--'  + f'C{k}', label=r'$\lambda^{an}$')
    ax[4, k].plot(i_sim, state.an.cl.ionomer_water_content, '-.' + f'C{k}', label=r'$\lambda^{an}_{ion,\text{CL}}$')
    ax[-1, k].set_xlabel('$i$ (A/cm$^2$)', fontsize=12)

ax[0, 0].set_ylabel('$V_{cell}$ (V)', fontsize=12)
ax[1, 0].set_ylabel('HFR (m$\Omega$.cm$^2$)', fontsize=12)
ax[2, 0].set_ylabel(r'$s_l^{ca}$ (%)', fontsize=12)
ax[3, 0].set_ylabel('$J_w$ (mmol/m$^2$.s)', fontsize=12)
ax[4, 0].set_ylabel('$\lambda$ (n.d.)', fontsize=12)

fig.tight_layout()

# Per-row legends, anchored to the right edge of each row's last axis
x_anchor = ax[0, -1].get_position().x1 + 0.01
for row in range(ax.shape[0]):
    handles, labels = ax[row, -1].get_legend_handles_labels()
    pos = ax[row, -1].get_position()
    y_mid = (pos.y0 + pos.y1) / 2
    fig.legend(handles=handles, labels=labels, loc='center left',
               bbox_to_anchor=(x_anchor, y_mid),
               bbox_transform=fig.transFigure, fontsize=8)

# plt.savefig(f'figures/internal-variables-{filename}.png', dpi=300, bbox_inches='tight')